# Multi-Agent Systems with LangGraph

## Overview

This notebook explores multi-agent systems where multiple specialized agents collaborate to solve complex problems.

## Learning Objectives

- Understand multi-agent architecture patterns
- Implement supervisor-worker pattern with LangGraph
- Configure agent-to-agent communication
- Manage shared state across agents
- Apply multi-agent systems to real-world scenarios

## Exam Objectives: 1.3, 1.5, 1.6
## Estimated Time: 75-90 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Architecture selection (reactive vs deliberative vs hybrid)
- ⭐⭐⭐ Trade-offs between architectures
- ⭐⭐ When to use each architecture pattern
- ⭐⭐ Scalability considerations

**Common Exam Scenarios:**
- Choosing architecture based on latency requirements
- Selecting architecture for different complexity levels
- Balancing speed vs sophistication

**Key Concepts to Master:**
- Reactive: Fast, rule-based, limited flexibility
- Deliberative: Complex reasoning, slower, goal-oriented
- Hybrid: Production-ready, balances both approaches

## Setup: Import Dependencies

In [ ]:
# Import core libraries
import os
from typing import Dict, List, Any, TypedDict, Annotated
from datetime import datetime

# Check for LangGraph (optional but recommended)
try:
    from langgraph.graph import StateGraph, END
    print("✅ LangGraph available")
    LANGGRAPH_AVAILABLE = True
except ImportError:
    print("ℹ️  LangGraph not available - using simplified examples")
    print("   Install with: pip install langgraph")
    LANGGRAPH_AVAILABLE = False

print("\n🎯 Setup complete!")

## Theory: Multi-Agent Architecture Patterns

### Why Multi-Agent Systems?

**Benefits:**
- **Specialization**: Each agent focuses on specific expertise
- **Scalability**: Distribute workload across agents
- **Modularity**: Easier to develop, test, and maintain
- **Robustness**: Failure of one agent doesn't crash the system

### Three Main Patterns

**1. Hierarchical (Supervisor-Worker)**
- Supervisor agent coordinates multiple worker agents
- Most common in production systems
- Clear delegation and coordination

**2. Peer-to-Peer (Collaborative)**
- Agents communicate directly without central coordinator
- Good for consensus-building
- More complex coordination

**3. Pipeline (Sequential)**
- Agents process tasks in fixed sequence
- Simple and predictable
- Good for ETL and content generation workflows

## Implementation: Simple Multi-Agent System

Let's implement a basic multi-agent system without LangGraph first.

In [ ]:
# Simple multi-agent system implementation

class Agent:
    """Base agent class."""
    
    def __init__(self, name: str, specialty: str):
        """
        Initialize agent.
        
        Args:
            name: Agent name
            specialty: Agent's area of expertise
        """
        self.name = name
        self.specialty = specialty
        self.inbox = []  # Messages received
    
    def process(self, task: str) -> str:
        """
        Process a task according to agent's specialty.
        
        Args:
            task: Task description
        
        Returns:
            Result of processing
        """
        # Simulate processing based on specialty
        return f"{self.name} ({self.specialty}): Processed '{task}'"
    
    def send_message(self, recipient: 'Agent', message: str):
        """
        Send message to another agent.
        
        Args:
            recipient: Agent to receive message
            message: Message content
        """
        recipient.inbox.append({
            "from": self.name,
            "message": message,
            "timestamp": datetime.now()
        })

class SupervisorAgent(Agent):
    """Supervisor agent that coordinates workers."""
    
    def __init__(self, name: str):
        """Initialize supervisor."""
        super().__init__(name, "coordination")
        # Track worker agents
        self.workers = {}
    
    def register_worker(self, worker: Agent):
        """
        Register a worker agent.
        
        Args:
            worker: Worker agent to register
        """
        self.workers[worker.specialty] = worker
        print(f"Registered worker: {worker.name} ({worker.specialty})")
    
    def delegate(self, task: str, specialty: str) -> str:
        """
        Delegate task to appropriate worker.
        
        Args:
            task: Task to delegate
            specialty: Required specialty
        
        Returns:
            Result from worker
        """
        # Find appropriate worker
        if specialty in self.workers:
            worker = self.workers[specialty]
            print(f"Supervisor delegating to {worker.name}...")
            return worker.process(task)
        else:
            return f"No worker available for specialty: {specialty}"
    
    def coordinate(self, user_request: str) -> Dict[str, str]:
        """
        Coordinate multiple workers to handle complex request.
        
        Args:
            user_request: User's request
        
        Returns:
            Dictionary of results from each worker
        """
        print(f"\nSupervisor received: {user_request}\n")
        
        # Decompose request into subtasks
        # In real system, LLM would do this analysis
        results = {}
        
        if "research" in user_request.lower():
            results["research"] = self.delegate(user_request, "research")
        
        if "analyze" in user_request.lower() or "analysis" in user_request.lower():
            results["analysis"] = self.delegate(user_request, "analysis")
        
        if "write" in user_request.lower() or "report" in user_request.lower():
            results["writing"] = self.delegate(user_request, "writing")
        
        return results

# Create multi-agent system
supervisor = SupervisorAgent("Coordinator")

# Create specialized worker agents
research_agent = Agent("ResearchBot", "research")
analysis_agent = Agent("AnalystBot", "analysis")
writing_agent = Agent("WriterBot", "writing")

# Register workers with supervisor
supervisor.register_worker(research_agent)
supervisor.register_worker(analysis_agent)
supervisor.register_worker(writing_agent)

# Test the system
print("\n" + "="*60)
results = supervisor.coordinate("Research AI agents, analyze the findings, and write a report")

print("\nResults:")
for specialty, result in results.items():
    print(f"  {specialty}: {result}")

## Implementation: Shared State Management

Agents often need to share state and information.

In [ ]:
# Shared state for multi-agent coordination

class SharedState:
    """Shared state accessible by all agents."""
    
    def __init__(self):
        """Initialize shared state."""
        # Store shared data
        self.data = {}
        # Track which agent updated what
        self.update_log = []
    
    def write(self, key: str, value: Any, agent_name: str):
        """
        Write data to shared state.
        
        Args:
            key: Data key
            value: Data value
            agent_name: Name of agent writing data
        """
        self.data[key] = value
        # Log the update
        self.update_log.append({
            "agent": agent_name,
            "key": key,
            "action": "write",
            "timestamp": datetime.now()
        })
        print(f"📝 {agent_name} wrote '{key}' to shared state")
    
    def read(self, key: str, agent_name: str) -> Any:
        """
        Read data from shared state.
        
        Args:
            key: Data key
            agent_name: Name of agent reading data
        
        Returns:
            Data value or None if not found
        """
        value = self.data.get(key)
        # Log the read
        self.update_log.append({
            "agent": agent_name,
            "key": key,
            "action": "read",
            "timestamp": datetime.now()
        })
        print(f"📖 {agent_name} read '{key}' from shared state")
        return value
    
    def get_all(self) -> Dict[str, Any]:
        """Get all shared data."""
        return self.data.copy()

class StatefulAgent(Agent):
    """Agent that uses shared state."""
    
    def __init__(self, name: str, specialty: str, shared_state: SharedState):
        """Initialize stateful agent."""
        super().__init__(name, specialty)
        # Reference to shared state
        self.shared_state = shared_state
    
    def process_with_state(self, task: str) -> str:
        """
        Process task using shared state.
        
        Args:
            task: Task to process
        
        Returns:
            Processing result
        """
        # Read from shared state
        previous_results = self.shared_state.read("previous_results", self.name)
        
        # Process task
        result = f"{self.name}: Processed '{task}'"
        
        if previous_results:
            result += f" (building on: {previous_results})"
        
        # Write result to shared state
        self.shared_state.write(f"{self.specialty}_result", result, self.name)
        
        return result

# Test shared state system
print("Testing Shared State System:\n")

shared_state = SharedState()

# Create agents with shared state
agent1 = StatefulAgent("Agent1", "research", shared_state)
agent2 = StatefulAgent("Agent2", "analysis", shared_state)

# Agents work sequentially, sharing state
result1 = agent1.process_with_state("Find information about AI")
print(f"Result 1: {result1}\n")

# Agent 2 can access Agent 1's results
shared_state.write("previous_results", result1, "System")
result2 = agent2.process_with_state("Analyze the findings")
print(f"Result 2: {result2}\n")

# View all shared state
print("Final Shared State:")
for key, value in shared_state.get_all().items():
    print(f"  {key}: {value}")

## Exercise: Design a Multi-Agent System

**Objective**: Design a multi-agent system for a specific use case.

**Scenario**: Create a content creation pipeline with three agents:
1. Research Agent: Gathers information
2. Writing Agent: Creates draft content
3. Editor Agent: Reviews and refines content

Implement the system using the patterns learned above.

In [ ]:
# Exercise: Content creation pipeline

class ContentPipeline:
    """Pipeline for content creation with multiple agents."""
    
    def __init__(self):
        """Initialize content pipeline."""
        self.shared_state = SharedState()
        
        # Create specialized agents
        self.research_agent = StatefulAgent("Researcher", "research", self.shared_state)
        self.writing_agent = StatefulAgent("Writer", "writing", self.shared_state)
        self.editor_agent = StatefulAgent("Editor", "editing", self.shared_state)
    
    def create_content(self, topic: str) -> Dict[str, str]:
        """
        Run content creation pipeline.
        
        Args:
            topic: Content topic
        
        Returns:
            Dictionary with results from each stage
        """
        print(f"Creating content about: {topic}\n")
        
        # Stage 1: Research
        print("Stage 1: Research")
        research_result = self.research_agent.process_with_state(f"Research {topic}")
        self.shared_state.write("research_findings", research_result, "Pipeline")
        print()
        
        # Stage 2: Writing
        print("Stage 2: Writing")
        research_findings = self.shared_state.read("research_findings", "Pipeline")
        writing_result = self.writing_agent.process_with_state(f"Write about {topic}")
        self.shared_state.write("draft_content", writing_result, "Pipeline")
        print()
        
        # Stage 3: Editing
        print("Stage 3: Editing")
        draft = self.shared_state.read("draft_content", "Pipeline")
        final_result = self.editor_agent.process_with_state(f"Edit content about {topic}")
        self.shared_state.write("final_content", final_result, "Pipeline")
        print()
        
        return {
            "research": research_result,
            "draft": writing_result,
            "final": final_result
        }

# Test the pipeline
pipeline = ContentPipeline()
results = pipeline.create_content("NVIDIA RAG Agents")

print("Pipeline Results:")
for stage, result in results.items():
    print(f"\n{stage.upper()}:")
    print(f"  {result}")

## Checkpoint: Self-Assessment

Before finishing, ensure you understand:

1. ✅ What are the three main multi-agent patterns?
   - Hierarchical (supervisor-worker), Peer-to-peer, Pipeline

2. ✅ When should you use multi-agent vs single-agent?
   - When task can be decomposed and requires specialization

3. ✅ What is shared state?
   - Common data structure accessible by all agents

4. ✅ What is the supervisor-worker pattern?
   - Central coordinator delegates tasks to specialized workers

5. ✅ What is the pipeline pattern best for?
   - Sequential processing where each stage depends on previous

## Next Steps

You've completed Module 1! Continue to:
- **Module 2**: Agent Development (prompt engineering, tool integration)
- **Module 5**: Cognition, Planning, and Memory (advanced reasoning)

## References

- Course Notes: module-01-agent-architecture-design.md (Section 4)
- LangGraph Documentation
- Multi-Agent Systems: A Modern Approach (Wooldridge)